# 15 Overview of measurements

<!-- navigation -->
[<kbd>← 13 Piezometer locations</kbd>](13%20Piezometer%20locations.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>20 Geohydrological scheme →</kbd>](20%20Geohydrological%20scheme.ipynb)

**Optional**: after the time series are imported (10, 11, 12) and notebook 13

Gives an overview of all time series per transect (period, time step, range, data availability, map), to check the data and choose the model period.

| | |
|---|---|
| **Input** | pastastore per transect |
| **Output** | overview table, figures in the transect folder 'timeseries'; candidate model periods in `results/model_period` |

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

import pandas as pd
tlt.show_versions()

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

In [ ]:
transects = project.transects()  # or a selection, e.g. ["Papendrecht West"]

## 1. Overview per transect

In [ ]:
rows = []
for transect in transects:
    pstore = project.open_pastastore(transect)
    names = pstore.oseries_names
    if not names:
        print(f"{transect}: the pastastore is empty")
        continue
    meta = project.transect_metadata(transect)
    for name in names:
        s = pstore.get_oseries(name).squeeze().dropna()
        rows.append({
            "transect": transect,
            "series": name,
            "kind": "river" if name == meta["loc_river"] else "piezometer",
            "start": s.index.min(),
            "end": s.index.max(),
            "values": len(s),
            "time_step_min": s.index.to_series().diff().median().total_seconds() / 60,
            "min": s.min(),
            "max": s.max(),
        })

    folder = project.transect_dir(transect, "timeseries")
    folder.mkdir(parents=True, exist_ok=True)
    title = f"{project.name} - {transect}"
    f, ax = tlt.plot_select(pstore, title=transect, subset="all series", figsize=(10, 6), units=units)
    tlt.finalize_and_save(f, folder, title, tag="series_all", file_label=transect)
    ax = pstore.plots.data_availability("oseries", set_yticks=True, figsize=(10, 4), progressbar=False)
    tlt.finalize_and_save(ax.figure, folder, title, tag="data_availability", file_label=transect)
    try:
        f, ax = tlt.plot_transect_map(project.piezometers(transect), line=project.crosssection(transect), crs=project.crs)
        tlt.finalize_and_save(f, folder, title, tag="piezometers_map", file_label=transect)
    except Exception as e:
        print(f"{transect}: no map ({type(e).__name__}: {e})")

overview = pd.DataFrame(rows).set_index(["transect", "series"])
overview.style.format(precision=2)

## 2. Candidate model periods
The model period of notebook 51 is a short period, e.g. a week, in which all piezometers used in the calibration (notebook 13: with `modellayer` and `x_crosssection`, and `use_calibration` not 0) and the river gauges have (nearly) complete data. What else makes a good period depends on the river, `mode`:

- `'tide'`, a river with tide: the tidal response is calibrated, so the daily mean of the river level and the heads should be fairly constant (no trend, see `check_trend` in notebook 51) and the river not high. The periods are ranked by the change of the daily mean (river + heads), smallest first.
- `'wave'`, a river without tide (e.g. held by weirs): the response to a high-water wave is calibrated, so the river level should change. The periods are ranked by the change of the daily mean of the river, largest first.

The daily mean is the 25-hour moving average, which filters out the tide. The figure shows the ranked candidates (green); the table lists them, and `candidate_model_periods.xlsx` all periods. Choose the model period from these and fill it in as `xlim` in notebook 51 (or as `PERIOD` in `run.py`, see `tlt.run_transects`).

In [ ]:
## UPDATE FOR YOUR PROJECT: 'tide' (a river with tide) or 'wave' (no tide: a high-water wave)
mode = "tide"
length = 7  # [days]

rivers, heads = project.model_period_series(transects)
periods = tlt.candidate_model_periods(rivers, heads, length=length, mode=mode)

folder = project.results_dir / "model_period"
folder.mkdir(parents=True, exist_ok=True)
f = tlt.plot_candidate_model_periods(rivers, heads, periods, units=units)
tlt.finalize_and_save(f, folder, project.name, tag="candidate_model_periods")
tlt.export_to_excel(periods, folder / "candidate_model_periods.xlsx", sheet_name="periods")

columns = ["end", "completeness", "least_complete", "river_change", "head_change", "river_high",
           "tidal_range", "score"]
periods.dropna(subset=["rank"]).sort_values("rank").set_index("rank", append=True)[columns].round(3)

## 3. Detail: one transect in a chosen period

In [ ]:
transect_name = transects[0]
xlim = None  # the complete series, or a part, e.g. [pd.Timestamp("2025-12-03"), pd.Timestamp("2025-12-05")]

pstore = project.open_pastastore(transect_name)
tlt.plot_select(pstore, xlim=xlim, title=transect_name, subset="all series" if xlim is None else "selected period", figsize=(10, 6), units=units);

## 4. Working with the pastastore directly
`project.open_pastastore(transect)` returns a `pastastore.PastaStore`, so you can also use [pastastore](https://pastastore.readthedocs.io) itself, e.g. to
look up the metadata of the series, get a series as a pandas Series or make the figures of pastastore. The
series are stored as `oseries`. A few examples for the transect of section 3:

In [ ]:
pstore.oseries  # metadata of all series: one row per series

In [ ]:
name = pstore.oseries_names[0]
series = pstore.get_oseries(name)  # a pandas Series with the measured heads
series.describe(percentiles=[0.1, 0.5, 0.9])

In [ ]:
# The figure of pastastore with one panel per series
pstore.plots.oseries(split=True, figsize=(10, 8), progressbar=False);

---

<!-- navigation -->
[<kbd>← 13 Piezometer locations</kbd>](13%20Piezometer%20locations.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>20 Geohydrological scheme →</kbd>](20%20Geohydrological%20scheme.ipynb)